# Financial Data ETL → RAG Gold Layer

A local, Python-only ETL pipeline that pulls **public financial data** (stock
prices via `yfinance` + real regulatory filings via the **SEC EDGAR** API),
cleans and joins it, and flattens it into natural-language documents ready to
feed into the RAG chatbot notebook.

**Personal, independent project** — built from scratch using public data only.
No proprietary code, data, or design from any employer.

Pipeline:
1. Install dependencies
2. Extract — stock price history + company info (`yfinance`)
3. Extract — company financial facts (SEC EDGAR API)
4. Clean & transform price data → derived metrics
5. Clean & transform SEC fundamentals → latest annual figures
6. Join price metrics + fundamentals + company info
7. Flatten into natural-language documents (the RAG "gold layer")
8. Write documents to `docs/` for the RAG notebook to ingest

> ⚠️ This notebook makes live network calls to Yahoo Finance and SEC EDGAR —
> run it on your own machine with internet access. It was not executed in the
> environment that generated it, so double-check the first run.

## Step 1 — Install dependencies

In [ ]:
!pip install -q yfinance pandas requests


## Step 2 — Configure companies to pull

Pick a handful of tickers — enough to demonstrate the pipeline, not a huge universe.

In [ ]:
TICKERS = ["AAPL", "MSFT", "JPM", "TSLA", "GOOGL"]

# SEC EDGAR requires every request to identify the requester via a User-Agent header.
# Replace with your own name/email — SEC blocks generic/missing User-Agents.
SEC_USER_AGENT = "Venugopal Data Engineering Project venugopal.example@email.com"


## Step 3 — Extract: stock price history + company info (`yfinance`)

In [ ]:
import yfinance as yf
import pandas as pd

raw_price_data = {}   # ticker -> price history DataFrame
raw_company_info = {} # ticker -> info dict

for ticker in TICKERS:
    t = yf.Ticker(ticker)
    raw_price_data[ticker] = t.history(period="1y")   # 1 year of daily prices
    raw_company_info[ticker] = t.info
    print(f"{ticker}: pulled {len(raw_price_data[ticker])} price rows")


## Step 4 — Extract: company financial facts (SEC EDGAR API)

SEC EDGAR keys companies by **CIK** (Central Index Key), not ticker, so we first
map tickers → CIKs using SEC's own public lookup file, then pull each company's
full financial facts JSON (this includes *every* filed figure, for every tag,
for every period — which is exactly the messy raw data we'll clean in Step 5).

In [ ]:
import requests
import time

headers = {"User-Agent": SEC_USER_AGENT}

# SEC's official ticker -> CIK mapping (public, no key needed)
ticker_map_resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=headers)
ticker_map_resp.raise_for_status()
ticker_map = ticker_map_resp.json()

# Build ticker -> zero-padded 10-digit CIK lookup
ticker_to_cik = {
    row["ticker"]: str(row["cik_str"]).zfill(10)
    for row in ticker_map.values()
}

raw_sec_facts = {}  # ticker -> raw company facts JSON

for ticker in TICKERS:
    cik = ticker_to_cik.get(ticker)
    if not cik:
        print(f"{ticker}: no CIK found, skipping SEC data")
        continue
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    resp = requests.get(url, headers=headers)
    if resp.status_code == 200:
        raw_sec_facts[ticker] = resp.json()
        print(f"{ticker}: pulled SEC facts (CIK {cik})")
    else:
        print(f"{ticker}: SEC request failed ({resp.status_code})")
    time.sleep(0.2)  # be polite to SEC's rate limits


## Step 5 — Clean & transform: price data → derived metrics

Raw daily price history isn't useful to hand an LLM directly — we compute a
few summary metrics per company instead.

In [ ]:
price_metrics = {}

for ticker, df in raw_price_data.items():
    if df.empty:
        print(f"{ticker}: no price data, skipping")
        continue

    df = df.dropna(subset=["Close"])
    latest_close = round(df["Close"].iloc[-1], 2)
    year_high = round(df["High"].max(), 2)
    year_low = round(df["Low"].min(), 2)
    start_close = round(df["Close"].iloc[0], 2)
    pct_change_1y = round(((latest_close - start_close) / start_close) * 100, 2)

    price_metrics[ticker] = {
        "latest_close": latest_close,
        "52w_high": year_high,
        "52w_low": year_low,
        "pct_change_1y": pct_change_1y,
    }

pd.DataFrame(price_metrics).T


## Step 6 — Clean & transform: SEC fundamentals → latest annual figures

The raw SEC JSON has every historical filing for every tag mixed together
(quarterly, annual, restatements, different fiscal-year-ends). We:
- Filter to just the tags we care about (Revenue, Net Income, Total Assets)
- Keep only **annual (10-K, form="10-K")** figures, not quarterly
- Take the **most recent** filed value per company
- Handle companies/tags that are missing gracefully instead of crashing

In [ ]:
TARGET_TAGS = {
    "Revenues": "revenue",
    "NetIncomeLoss": "net_income",
    "Assets": "total_assets",
}

fundamentals = {}

for ticker, facts in raw_sec_facts.items():
    company_fundamentals = {}
    us_gaap = facts.get("facts", {}).get("us-gaap", {})

    for sec_tag, clean_name in TARGET_TAGS.items():
        tag_data = us_gaap.get(sec_tag)
        if not tag_data:
            company_fundamentals[clean_name] = None
            continue

        usd_values = tag_data.get("units", {}).get("USD", [])
        # Keep only annual 10-K filings
        annual_values = [v for v in usd_values if v.get("form") == "10-K"]
        if not annual_values:
            company_fundamentals[clean_name] = None
            continue

        # Most recently filed value
        latest = max(annual_values, key=lambda v: v.get("end", ""))
        company_fundamentals[clean_name] = {
            "value": latest.get("val"),
            "fiscal_year_end": latest.get("end"),
        }

    fundamentals[ticker] = company_fundamentals

fundamentals


## Step 7 — Join: price metrics + fundamentals + company info

Merge everything into one record per company, keyed by ticker.

In [ ]:
joined_records = {}

for ticker in TICKERS:
    info = raw_company_info.get(ticker, {})
    joined_records[ticker] = {
        "ticker": ticker,
        "company_name": info.get("longName", ticker),
        "sector": info.get("sector", "Unknown"),
        "industry": info.get("industry", "Unknown"),
        "market_cap": info.get("marketCap"),
        "price": price_metrics.get(ticker, {}),
        "fundamentals": fundamentals.get(ticker, {}),
    }

joined_records[TICKERS[0]]  # peek at one record


## Step 8 — Flatten into natural-language documents (the RAG gold layer)

This is the step that turns a joined structured record into something an
embedding model can meaningfully retrieve — one short paragraph per company.

In [ ]:
def format_currency(value):
    if value is None:
        return "not available"
    return f"${value:,.0f}"


def build_company_document(record: dict) -> str:
    price = record["price"]
    fund = record["fundamentals"]

    revenue = fund.get("revenue")
    net_income = fund.get("net_income")
    assets = fund.get("total_assets")

    doc = f"""Company: {record['company_name']} ({record['ticker']})
Sector: {record['sector']} | Industry: {record['industry']}
Market Cap: {format_currency(record['market_cap'])}

Stock Performance (trailing 12 months):
Latest close price: ${price.get('latest_close', 'N/A')}
52-week high: ${price.get('52w_high', 'N/A')} | 52-week low: ${price.get('52w_low', 'N/A')}
1-year price change: {price.get('pct_change_1y', 'N/A')}%

Financials (most recent annual 10-K filing):
Revenue: {format_currency(revenue['value']) if revenue else 'not available'} (fiscal year ending {revenue['fiscal_year_end'] if revenue else 'N/A'})
Net Income: {format_currency(net_income['value']) if net_income else 'not available'} (fiscal year ending {net_income['fiscal_year_end'] if net_income else 'N/A'})
Total Assets: {format_currency(assets['value']) if assets else 'not available'} (fiscal year ending {assets['fiscal_year_end'] if assets else 'N/A'})
"""
    return doc.strip()


documents = {ticker: build_company_document(record) for ticker, record in joined_records.items()}

print(documents[TICKERS[0]])


## Step 9 — Write documents to `docs/`

These `.txt` files are ready to drop straight into the `docs/` folder your
RAG chatbot notebook reads from (Step 2 of that notebook).

In [ ]:
import os

OUTPUT_FOLDER = "docs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for ticker, doc_text in documents.items():
    filename = os.path.join(OUTPUT_FOLDER, f"{ticker}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(doc_text)

print(f"Wrote {len(documents)} documents to '{OUTPUT_FOLDER}/':")
print(os.listdir(OUTPUT_FOLDER))


## Next steps

- Copy the `docs/` folder this notebook produced into your RAG chatbot project
  (or upload its files via the Colab upload cell in that notebook).
- Run the RAG notebook as before — chunking, embedding, and retrieval will
  now work over these company documents instead of the sample docs.
- Try questions like: *"What is Apple's revenue?"*, *"Compare Tesla and
  Microsoft's stock performance"*, *"Which company has the highest market cap?"*
- **For your portfolio/interview writeup:** the ETL story here is the SEC data
  cleaning (Step 6) — raw nested JSON with every historical filing mixed
  together, filtered down to clean annual figures. That's the concrete,
  defensible "I did real ETL" detail to describe.